<a href="https://colab.research.google.com/github/hirototoda/garbage_collect/blob/main/yolo11_roboflow_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install "ultralytics<=8.3.40" supervision roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.2/112.6 GB disk)


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XbUFsR4k7rv2ghtTdxCT")
project = rf.workspace("programming1bworkspace").project("programming-1b-first")
version = project.version(2)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 0.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to programming-1b-first-2 in yolov11:: 100%|██████████| 234/234 [00:00<00:00, 6305.93it/s]


### データセット検査 & `data.yaml` 自動補正

val 時の `IndexError: index N is out of bounds ...`(`ConfusionMatrix.process_batch` 内)は、`data.yaml` の `nc`/`names` より大きなクラス ID がラベルファイルに含まれているときに発生します。ここで `data.yaml` と実ラベルのクラス ID を突き合わせ、ズレていれば `nc`/`names` を実ラベルに合わせて上書きします。

In [ ]:
import os, glob, yaml, shutil

yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(yaml_path) as f:
    data = yaml.safe_load(f)
print('data.yaml (before):', data)

def classes_in_split(split):
    label_dir = os.path.join(dataset.location, split, 'labels')
    if not os.path.isdir(label_dir):
        return None
    found = set()
    for lf in glob.glob(os.path.join(label_dir, '*.txt')):
        with open(lf) as fh:
            for line in fh:
                parts = line.strip().split()
                if parts:
                    found.add(int(parts[0]))
    return found

found = {s: classes_in_split(s) for s in ('train', 'valid', 'test')}
print('classes in labels:', found)

all_classes = set().union(*(c for c in found.values() if c))
declared_nc = data.get('nc') or len(data.get('names') or [])
max_cls = max(all_classes) if all_classes else -1

if max_cls + 1 > declared_nc:
    new_nc = max_cls + 1
    names = list(data.get('names') or [])
    while len(names) < new_nc:
        names.append(f'class_{len(names)}')
    data['nc'] = new_nc
    data['names'] = names[:new_nc]
    shutil.copy(yaml_path, yaml_path + '.bak')
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    print('data.yaml (after):', data)
else:
    print('data.yaml already consistent with labels - no fix needed')

In [ ]:
!yolo task=detect \
mode=train \
model=yolo11s.pt \
data={dataset.location}/data.yaml \
epochs=10 \
imgsz=640 \
plots=True

New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolo11s.pt, data=/content/programming-1b-first-2/data.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fa

In [ ]:
!yolo task=detect \
mode=val \
model=runs/detect/train/weights/best.pt \
data={dataset.location}/data.yaml

Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Scanning /content/programming-1b-first-2/valid/labels.cache... 30 images, 0 backgrounds, 0 corrupt: 100% 30/30 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  50% 1/2 [00:01<00:01,  1.30s/it]
Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 972, in entrypoint
    getattr(model, mode)(**overrides)  # default args from model
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 638, in val
    validator(model=self.model)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context

In [ ]:
import os
# Roboflow の分割によっては test/images が無いので、無ければ valid/images を使う。
src_dir = f'{dataset.location}/test/images'
if not os.path.isdir(src_dir):
    src_dir = f'{dataset.location}/valid/images'
print('predict source:', src_dir)

!yolo task=detect \
mode=predict \
model=runs/detect/train/weights/best.pt \
conf=0.25 \
source={src_dir}